# Opinion Evolution Tracker — Colab Training

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

This notebook runs everything that could not be run locally (development happened in a
bandwidth-constrained environment where `bert-base-multilingual-cased` / `xlm-roberta-base`
could not be downloaded — tokenizer files came through but the ~680MB weights file stalled
at 0 bytes after 8+ minutes). All code has already been proven correct end-to-end on real
data using a small already-cached substitute encoder
(`scripts/sanity_check_pipeline.py`, `scripts/train.py`, `scripts/cross_domain_eval.py` were
all run successfully that way). This notebook repeats the same commands with the real
encoders, which Colab's bandwidth can actually fetch.

Order matters: full-model training (cell 7) must complete for both `amazon` and one
`dravidian` language **before** cross-domain evaluation (cell 9), which loads both
checkpoints at once.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repository

In [ ]:
!git clone https://github.com/prakyath006/Tracking-Opinion-Evolution-in-Multilingual-Sequential-Text.git
%cd Tracking-Opinion-Evolution-in-Multilingual-Sequential-Text

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Get the data

`data/` is gitignored (raw + preprocessed files are hundreds of MB). Upload it as a zip via
the Colab file browser, or mount Drive if you already keep a copy there. Either way it must
end up at `data/raw/` and `data/preprocessed/` under the repo root, matching what
`src/dataset.py` and `src/preprocessing.py` already expect.

In [ ]:
# Option A: upload data.zip via the Colab file browser, then:
# !unzip -q data.zip -d .

# Option B: mount Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/<your_data_folder>/* data/

!ls data/raw/ data/preprocessed/

## 5. Verify everything is wired correctly before spending GPU time

In [ ]:
!python -m pytest tests/ -q
!python scripts/verify_guide_modules.py

## 6. Full pipeline sanity check with the real encoder

One forward pass per domain, no training — confirms bert-base-multilingual-cased actually
downloads and runs here (it could not be downloaded in the dev environment).

In [ ]:
!python scripts/sanity_check_pipeline.py --model_name bert-base-multilingual-cased

## 7. Train the full model (OpinionEvolutionTracker)

Each run writes `outputs/checkpoints/best_model_<run_id>.pt` — domain-specific filenames,
so training Amazon then Dravidian does not overwrite the earlier checkpoint (this used to be
a single shared `best_model.pt`, which made cross-domain evaluation of both directions
impossible). Class weights are computed automatically from each run's own training split
(`--auto_class_weights`, on by default) to counteract the trajectory-head imbalance seen on
every domain (STABLE is 56-84% of sequences everywhere; Dravidian's IMPROVING/DECLINING
classes are only 1-2% of sequences).

**Runtime estimate** (measured locally with a much smaller substitute encoder, scaled up):
a full epoch over Amazon's 6,411 training sequences took 147s with a 6-layer/384-dim
encoder (25.7M params total, frozen) on a modest mobile GPU (RTX 3050, 4GB). mBERT
(110M params, 12 layers, 768-dim) is roughly 4x larger; a Colab T4 is faster than that
mobile GPU. Expect on the order of **10-20 minutes/epoch** for Amazon, less for the smaller
Dravidian languages (malayalam: 3,385 train sequences, kannada: 1,216) with default
settings (`--epochs 20 --patience 5`, so actual wall-clock depends on when early stopping
triggers -- expect roughly 1-4 hours per domain, not the full 20 epochs).

In [ ]:
!python scripts/train.py --domain amazon --epochs 20 --batch_size 16

In [ ]:
!python scripts/train.py --domain dravidian --language tamil --epochs 20 --batch_size 16

In [ ]:
# Optional: the other two Dravidian languages, for a fuller cross-domain matrix.
!python scripts/train.py --domain dravidian --language malayalam --epochs 20 --batch_size 16
!python scripts/train.py --domain dravidian --language kannada --epochs 20 --batch_size 16

## 8. Train baselines (Guide Module 3 + ablation, Steps 5 & 7)

`scripts/train_baselines.py` covers all 5 models in `BASELINE_REGISTRY`:

- `mbert_sentence`, `xlmr_sentence`, `textcnn` — single-review classifiers, no sequence
  modeling at all. Trained on flattened (review, sentiment) pairs, scored on sentiment only.
- `lstm_only` (Bi-LSTM, no attention) and `attention_only` (attention, no Bi-LSTM) —
  together with the full model (Bi-LSTM **and** attention) these three variants isolate
  each component's contribution: full vs lstm_only shows what attention buys you, full vs
  attention_only shows what the Bi-LSTM buys you. This is the ablation study (Step 7) — no
  additional variants were needed beyond what `src/baselines.py` already defines.

In [ ]:
!python scripts/train_baselines.py --baseline all --domain amazon --epochs 10 --batch_size 16

In [ ]:
!python scripts/train_baselines.py --baseline all --domain dravidian --language tamil --epochs 10 --batch_size 16

## 9. Cross-domain evaluation (both directions, Step 8)

Requires the `amazon` and `dravidian_tamil` checkpoints from Section 7. Evaluates:
Amazon-trained model on {Amazon, Tamil, Malayalam, Kannada} test sets, AND
Tamil-trained model on {Tamil, Amazon, Malayalam, Kannada} test sets — the full matrix,
not just Amazon-to-Dravidian.

In [ ]:
!python scripts/cross_domain_eval.py --language tamil

## 10. Compile the full metrics table (Step 9)

Gathers every result written above — full model in-domain, all baselines, cross-domain
both directions — into one comparison table (Accuracy/Precision/Recall/F1 per task head,
plus SCS) at `outputs/metrics/results_table.{csv,md,json}`.

In [ ]:
!python scripts/compile_metrics.py

## 11. Download everything

In [ ]:
from google.colab import files
!zip -r outputs.zip outputs/
files.download('outputs.zip')